# Lectures 24–27 — Google Earth Engine for Urban Spatial Analysis

**HS524M — Urban Informatics and Spatial Analytics**

These three lectures introduce **Google Earth Engine (GEE)** using Python and Jupyter.

The lectures follow one urban-environmental analysis sequence:

- **Lectures 24 and 25:** Google Earth Engine basics + vegetation mapping using NDVI
- **Lectures 26 and 27:** Urban heat + population exposure

The emphasis is not on remote sensing for its own sake. The objective is to use large geospatial datasets to answer urban questions.

Throughout the notebook, we use **Bengaluru** as a case study.


## Before starting

Earth Engine is different from the local GIS workflow used in the earlier lectures.

Earlier:

```text
Download data
      ↓
Store data on our computer
      ↓
Read with GeoPandas / Rasterio
      ↓
Process locally
```

With Earth Engine:

```text
Earth Engine Data Catalog
      ↓
Select data on Google's servers
      ↓
Filter and process on the server
      ↓
Visualise or request the result we need
```

This means we can work with very large satellite archives without manually downloading every image.


# Lectures 24 and 25 — Introduction to Google Earth Engine and NDVI

In this lecture we will:

1. set up the Earth Engine Python environment;
2. authenticate and initialise Earth Engine;
3. understand the main Earth Engine data objects;
4. explore the Earth Engine Data Catalog;
5. work with a Sentinel-2 `ImageCollection`;
6. filter satellite imagery by location, date, and cloud cover;
7. apply a pixel-level cloud mask;
8. create a composite image;
9. calculate NDVI; and
10. display the results interactively using `geemap`.




## 1. One-time installation in the conda environment

Run the following command in **Anaconda Prompt / Miniforge Prompt**, after activating the conda environment used for this course:

```text
conda install -c conda-forge earthengine-api geemap
```

We are installing two important packages:

- `earthengine-api` — allows Python to communicate with Google Earth Engine;
- `geemap` — provides convenient interactive maps and additional tools for using Earth Engine from Jupyter.

This installation normally needs to be done only once for the environment.


## 2. Project ID used in this notebook

The instructor's Google Cloud / Earth Engine project ID is:

`uisa-class`

Earth Engine requires an enabled and registered Google Cloud project.

**Important for students:** if you are not granted access to `uisa-class`, replace the project ID below with the project ID that you created and registered for Earth Engine.


In [1]:
import ee
import geemap

PROJECT_ID = "solar-grail-510708-p8"

print("Project ID:", PROJECT_ID)

Project ID: solar-grail-510708-p8


## 3. Authenticate Earth Engine

Authentication connects the Python environment to your Google account.

The first time you run `ee.Authenticate()`, a browser window or authentication link may appear. Sign in using the Google account that has Earth Engine access.

You normally do **not** need to authenticate every time you run a notebook. Once valid credentials exist, Earth Engine can reuse them.


In [2]:
ee.Authenticate()

True

## 4. Initialise Earth Engine

Authentication answers:

> **Who are you?**

Initialisation answers:

> **Which Earth Engine-enabled Google Cloud project should this Python session use?**

We initialise using the project ID `uisa-class`.


In [3]:
PROJECT_ID

'solar-grail-510708-p8'

In [4]:
ee.Initialize(project= PROJECT_ID)

print("Google Earth Engine initialized successfully.")

EEException: Caller does not have required permission to use project solar-grail-510708-p8. Grant the caller the roles/serviceusage.serviceUsageConsumer role, or a custom role with the serviceusage.services.use permission, by visiting https://console.developers.google.com/iam-admin/iam?project=solar-grail-510708-p8 and then retry. Propagation of the new permission may take a few minutes.

### A quick connection test

Most Earth Engine objects live on Google's servers.

In the following example, `ee.String(...)` creates an Earth Engine object. `.getInfo()` requests its value and brings the small result back to local Python.


In [ ]:
message = ee.String("Connected to Google Earth Engine")

print(message.getInfo())

## 5. Server-side versus local objects

This is one of the most important ideas in Earth Engine.

When we write:

```python
image = ee.Image(...)
```

the actual raster is **not normally downloaded to our computer**.

Instead, Python creates a description of the operation that Earth Engine should perform on the server.

So we can think of it as:

```text
Python / Jupyter
      ↓
instructions
      ↓
Google Earth Engine servers
      ↓
processing of very large geospatial datasets
      ↓
map / statistic / exported result
```

`getInfo()` should therefore be used mainly for **small outputs**, such as counts, metadata, or summary statistics. We should not try to use it to download a large raster into Python.


## 6. Main Earth Engine object types

Four object types will appear repeatedly.

### `ee.Image`

One raster image.

Examples:

- one satellite scene;
- one NDVI raster;
- one land-surface-temperature raster.

### `ee.ImageCollection`

A collection of many images.

For example, Sentinel-2 imagery for Bengaluru over an entire year is an `ImageCollection`.

### `ee.Feature`

One vector feature.

For example, one ward polygon.

### `ee.FeatureCollection`

A collection of vector features.

For example, all ward polygons in a city.

A useful parallel with what we already know is:

```text
GeoPandas                         Earth Engine
---------------------------------------------------------
one geometry                      ee.Feature
GeoDataFrame                      ee.FeatureCollection

one raster                        ee.Image
many rasters through time         ee.ImageCollection
```


## 7. Earth Engine Data Catalog

Earth Engine contains a large catalog of ready-to-use geospatial datasets.

Examples relevant to urban analysis include:

| Urban question | Example dataset | Earth Engine ID |
|---|---|---|
| Vegetation / land surface | Sentinel-2 Surface Reflectance | `COPERNICUS/S2_SR_HARMONIZED` |
| Surface temperature | Landsat 8 Level-2 | `LANDSAT/LC08/C02/T1_L2` |
| Surface temperature | Landsat 9 Level-2 | `LANDSAT/LC09/C02/T1_L2` |
| Population exposure | WorldPop | `WorldPop/GP/100m/pop` |
| Atmospheric NO₂ | Sentinel-5P | `COPERNICUS/S5P/OFFL/L3_NO2` |

The dataset ID is important because this is what we use in Python to access the collection.


## 8. Define a study area

We can either use a rectangular area covering Bengaluru. Or a geodataframe

The file is located at:
C:\Teaching\IIT Tirupati\UISA\2026\uisa_codes\data\Bengaluru_boundary.geojson

We first read it using GeoPandas and then convert it into an Earth Engine object.


In [ ]:
import geopandas as gpd

data_path = r"C:\Teaching\IIT Tirupati\UISA\2026\uisa_codes\data"

bengaluru_gdf = gpd.read_file(
    data_path + r"\Bengaluru_boundary.geojson"
)

bengaluru_gdf

The boundary is currently a GeoPandas GeoDataFrame stored locally on our computer.

Google Earth Engine cannot directly use a GeoDataFrame. We therefore convert it into an Earth Engine `FeatureCollection`.

`geemap` provides a convenient function called `geopandas_to_ee()` for this conversion.

In [ ]:
bengaluru = geemap.geopandas_to_ee(
    bengaluru_gdf
)

bengaluru

### Display the study area

`geemap.Map()` creates an interactive map inside Jupyter.

We can add Earth Engine objects directly to this map.


In [ ]:
Map = geemap.Map()

Map.centerObject(bengaluru, 10) ## conrtols zoom

Map.addLayer(
    bengaluru,
    {"color": "blue"},
    "Bengaluru Boundary"
)

Map

## 9. Sentinel-2 Surface Reflectance

For vegetation analysis, we use:

`COPERNICUS/S2_SR_HARMONIZED`

Sentinel-2 provides multiple spectral bands.

For today's analysis, the most important bands are:

| Band | Meaning | Approx. pixel size |
|---|---|---:|
| `B2` | Blue | 10 m |
| `B3` | Green | 10 m |
| `B4` | Red | 10 m |
| `B8` | Near Infrared (NIR) | 10 m |
| `SCL` | Scene Classification Layer | 20 m |

For NDVI we need:

- **Red** → `B4`
- **Near Infrared** → `B8`


## 10. Load the Sentinel-2 ImageCollection

We first access the complete Sentinel-2 Surface Reflectance collection.

At this point, the collection contains images from many places and many dates.

We therefore need to filter it.


In [ ]:
s2_all = ee.ImageCollection(
    "COPERNICUS/S2_SR_HARMONIZED"
)

s2_all

## Understanding Google Earth Engine Dataset IDs

Google Earth Engine provides access to a very large **Data Catalog** containing satellite imagery, climate data, population data, elevation models, atmospheric products, land-cover data, and many other geospatial datasets.

Each dataset has a unique **Earth Engine ID**.

For example:

`COPERNICUS/S2_SR_HARMONIZED`

can be understood as:

- `COPERNICUS` → Copernicus Earth observation programme
- `S2` → Sentinel-2
- `SR` → Surface Reflectance
- `HARMONIZED` → harmonized collection designed to keep the dataset more consistent through time

In Python, we use this ID to access the dataset:

`ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")`

This does not download the entire dataset. It creates a connection to the Earth Engine collection stored on Google's servers.

### How to find dataset IDs

1. Open the **Google Earth Engine Data Catalog**.
2. Search for the dataset you need, for example:
   - Sentinel-2
   - Landsat
   - population
   - elevation
   - NO₂
3. Open the dataset page.
4. Look for the **Earth Engine Snippet** or dataset ID.
5. Copy the ID and use it in Python.

For example:

`COPERNICUS/S2_SR_HARMONIZED`

or:

`LANDSAT/LC09/C02/T1_L2`

or:

`COPERNICUS/S5P/OFFL/L3_NO2`

### Before using any dataset

Always check the dataset documentation for:

- available bands;
- band names;
- spatial resolution;
- temporal coverage;
- units;
- scale factors;
- NoData or quality information;
- cloud or quality-mask bands; and
- any important limitations.

The dataset ID tells Earth Engine **which collection to use**, but the documentation tells us **how to use it correctly**.

## 11. Filter by location, date, and cloudiness

We will use imagery from the full year **2025**.

The sequence is:

```text
All Sentinel-2 images
        ↓
filterBounds(Bengaluru)
        ↓
filterDate(2025)
        ↓
filter images with high overall cloudiness
```

`CLOUDY_PIXEL_PERCENTAGE` is an image-level property. It helps remove highly cloudy scenes before we do more detailed pixel-level cloud masking.


In [ ]:
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(bengaluru) # Keep only images that overlap the Bengaluru boundary
    .filterDate("2025-01-01", "2026-01-01") # date filter
    .filter(
        ee.Filter.lt(
            "CLOUDY_PIXEL_PERCENTAGE",
            30
        )
    )
) ## keep images where the cloudy pixels is less than 30%; lt stands for 'less than'

print(
    "Number of Sentinel-2 images after filtering:",
    s2.size().getInfo()
)

## 12. Why do we still need pixel-level cloud masking?

Filtering with:

```python
CLOUDY_PIXEL_PERCENTAGE < 30
```

removes images that are very cloudy overall.

But an image with 20% cloud cover can still contain clouds over our study area.

Therefore we also remove cloudy pixels.

Sentinel-2 Surface Reflectance contains an `SCL` — **Scene Classification Layer**.

Important SCL classes include:

- `3` = cloud shadow
- `7` = low-probability cloud / unclassified
- `8` = medium-probability cloud
- `9` = high-probability cloud
- `10` = cirrus
- `11` = snow / ice

We mask these classes before creating the composite.


In [ ]:
def mask_s2_clouds(image):

    scl = image.select("SCL")

    clear = (
        scl.neq(3) ## neq stands for not equal to 
        .And(scl.neq(7)) ## we are filtering out cloudy, shadowy pixels
        .And(scl.neq(8)) ## .And() is EE's pixel-wise logical operation for image objects.
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    reflectance = (
        image
        .select(["B2", "B3", "B4", "B8"])
        .multiply(0.0001) #converts the stored integer-like values into reflectance values
    )

    return (
        reflectance
        .updateMask(clear)
        .copyProperties(
            image,
            ["system:time_start"]
        )
    )

### Understanding the cloud-mask function

For every Sentinel-2 image:

1. select the `SCL` band;
2. identify pixels that are not cloud, cloud shadow, cirrus, or snow;
3. keep the four reflectance bands needed for our analysis;
4. apply the Sentinel-2 reflectance scale factor (`0.0001`);
5. mask unwanted pixels; and
6. retain the acquisition-time information.

The function itself does not yet process the entire collection.

We apply it to each image using `.map()`.


In [ ]:
s2_clean = s2.map(mask_s2_clouds)

s2_clean

## 13. From many images to one composite

We still have an `ImageCollection` containing many cloud-masked images.

For mapping vegetation across the year, we can create a **median composite**:

```python
collection.median()
```

For every location, Earth Engine finds the median value across the available images.

This reduces the influence of remaining unusual observations and gives us one representative image.


In [ ]:
s2_median = (
    s2_clean
    .median()
    .clip(bengaluru)
)

print(
    "Composite bands:",
    s2_median.bandNames().getInfo()
)

In [ ]:
s2_median

## 14. True-colour image

Before calculating NDVI, it is useful to inspect the imagery visually.

A true-colour image uses:

- Red → `B4`
- Green → `B3`
- Blue → `B2`

This approximately reproduces the colours that the human eye would see.


In [ ]:
## visualization setting for the bands 
true_color_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0.0,
    "max": 0.30
}

Map = geemap.Map()

Map.centerObject(bengaluru, 10)

Map.addLayer(
    s2_median,
    true_color_vis,
    "Sentinel-2 true colour"
)

Map.addLayer(
    bengaluru,
    {"color": "white"},
    "Study area"
)

Map

## 15. NDVI — Normalized Difference Vegetation Index

## NDVI — Normalized Difference Vegetation Index

Healthy vegetation absorbs much of the **red light** reaching it and reflects strongly in the **near-infrared (NIR)** region.

NDVI uses this contrast between red and near-infrared reflectance.

**NDVI formula:**

$$
NDVI = \frac{NIR - Red}{NIR + Red}
$$

For Sentinel-2:

- `B8` = Near Infrared (NIR)
- `B4` = Red

Therefore, for Sentinel-2:

$$
NDVI = \frac{B8 - B4}{B8 + B4}
$$

NDVI generally ranges from approximately `-1` to `+1`.

A simple interpretation is:

```text
Lower or negative values  → water, built-up surfaces, bare areas, etc.

Higher positive values    → stronger vegetation signal

In [ ]:
ndvi = (
    s2_median
    .normalizedDifference(
        ["B8", "B4"]
    )
    .rename("NDVI")
)

ndvi

### Why use `normalizedDifference()`?

Earth Engine provides a convenient function called `normalizedDifference()` for indices that follow the form:

$$
\frac{A - B}{A + B}
$$

For NDVI, we use:

- `B8` = Near Infrared (NIR)
- `B4` = Red

So:

```python
normalizedDifference(["B8", "B4"])

In [ ]:
import geemap.colormaps as cm

ndvi_vis = {
    "min": 0,
    "max": 1,
    "palette": cm.palettes.ndvi
}

Map = geemap.Map()

Map.centerObject(bengaluru, 10)

Map.addLayer(
    s2_median,
    true_color_vis,
    "True colour"
)

Map.addLayer(
    ndvi,
    ndvi_vis,
    "NDVI"
)

Map.addLayer(
    bengaluru,
    {"color": "white"},
    "Study area"
)

Map

## 16. Summarising a raster with a reducer

Earth Engine **reducers** summarise raster values.

For example:

```python
ee.Reducer.mean()
```

calculates the mean.

Here we calculate mean NDVI for the study area.

`scale=10` asks Earth Engine to perform the operation at approximately the 10 m Sentinel-2 scale.


In [ ]:
mean_ndvi = ndvi.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=bengaluru,
    scale=10,
    maxPixels=1e9
)

print(
    "Mean NDVI:",
    mean_ndvi.get("NDVI").getInfo()
)

In [ ]:
# Add a colour bar
Map.add_colorbar(
    ndvi_vis,
    label="NDVI",
    layer_name="NDVI",
    orientation="vertical",
    position="bottomright"
)

# Add a title
Map.add_text(
    "NDVI of Bengaluru — 2025",
    fontsize=20,
    bold=True,
    bg_color="white",
    position="topright"
)

Map

## Lectures 24 and 25 — Main workflow

The sequence used today is a general Earth Engine pattern:

```text
Select dataset
      ↓
Filter location
      ↓
Filter date
      ↓
Filter image quality
      ↓
Mask unwanted pixels
      ↓
Create composite
      ↓
Calculate an indicator
      ↓
Visualise
      ↓
Summarise
```

The same logic will be reused in Lectures 25 and 26.


# Lecture 26 and 27 — Urban Heat and Population Exposure

The question now changes from:

> **Where is vegetation?**

to:

> **Where is the urban surface hottest, and how many people live in the hottest areas?**

This introduces an important conceptual distinction:

```text
HAZARD
Where is it hot?
        +
EXPOSURE
Where are people?
        ↓
POPULATION EXPOSED TO HIGH SURFACE TEMPERATURE
```

We use:

- **Landsat 8 and Landsat 9 Level-2** surface-temperature data; and
- **WorldPop** gridded population data.


## 1. Land Surface Temperature is not air temperature

**Land Surface Temperature (LST)** measures the temperature of the Earth's surface as observed by the satellite.

Examples include the temperature of:

- roofs;
- roads;
- bare soil;
- vegetation;
- water; and
- other surfaces.

It is **not the same as the air temperature** reported by a weather station.

This distinction is important when interpreting an urban heat map.


## 2. Why Landsat?

Sentinel-2 is excellent for vegetation because it provides 10 m optical bands.

However, Sentinel-2 does not provide the thermal band needed for LST.

Landsat 8 and 9 provide a Level-2 **surface-temperature band**:

`ST_B10`

The Earth Engine catalog gives the following conversion:

```text
Temperature in Kelvin
=
ST_B10 × 0.00341802 + 149
```

To convert Kelvin to degrees Celsius:

```text
°C = Kelvin - 273.15
```

The surface-temperature product is approximately 30 m in the Earth Engine Level-2 dataset.


## 3. Load Landsat 8 and Landsat 9

We combine the two collections to increase the number of observations.

For this example, we study Bengaluru during the **pre-monsoon hot season: March–May 2025**.


In [ ]:
landsat8 = ee.ImageCollection(
    "LANDSAT/LC08/C02/T1_L2"
)

landsat9 = ee.ImageCollection(
    "LANDSAT/LC09/C02/T1_L2"
)

landsat = landsat8.merge(
    landsat9
)

landsat_hot_season = (
    landsat
    .filterBounds(bengaluru)
    .filterDate(
        "2025-03-01",
        "2025-06-01"
    )
    .filter(
        ee.Filter.lt(
            "CLOUD_COVER",
            50
        )
    )
    .filter(
        ee.Filter.eq(
            "PROCESSING_LEVEL",
            "L2SP"
        )
    )
)

print(
    "Landsat scenes:",
    landsat_hot_season.size().getInfo()
)

## Landsat Processing Levels

Satellite images go through different levels of processing before they are ready for analysis.

A simple way to understand the progression is:

```text
Satellite sensor
      ↓
Level 1
Radiometrically and geometrically processed observations
      ↓
Level 2
Derived physical surface products
      ↓
Surface Reflectance
Surface Temperature

## 4. Cloud masking using `QA_PIXEL`

Landsat provides a `QA_PIXEL` quality-assurance band.

Different bits identify conditions such as:

- dilated cloud;
- cirrus;
- cloud;
- cloud shadow; and
- snow.

We keep pixels where these unwanted conditions are absent.

The function also converts the thermal band to degrees Celsius.


In [ ]:
def prepare_landsat_lst(image):

    qa = image.select("QA_PIXEL")

    clear = (
        qa.bitwiseAnd(1 << 1).eq(0)
        .And(
            qa.bitwiseAnd(1 << 2).eq(0)
        )
        .And(
            qa.bitwiseAnd(1 << 3).eq(0)
        )
        .And(
            qa.bitwiseAnd(1 << 4).eq(0)
        )
        .And(
            qa.bitwiseAnd(1 << 5).eq(0)
        )
    )

    lst_c = (
        image
        .select("ST_B10")
        .multiply(0.00341802)
        .add(149.0)
        .subtract(273.15)
        .rename("LST_C")
    )

    return (
        lst_c
        .updateMask(clear)
        .copyProperties(
            image,
            ["system:time_start"]
        )
    )

### Understanding `QA_PIXEL` and `1 << 3`

The `QA_PIXEL` band does not store reflectance or temperature.

Instead, it stores **quality information about each pixel**, such as whether the pixel contains:

- cloud
- cloud shadow
- cirrus
- snow

Rather than storing each of these conditions in a separate band, Landsat stores several **yes/no switches inside a single number**.

Think of the QA value as a row of switches:

    Bit position:

    5       4        3       2        1
    ↓       ↓        ↓       ↓        ↓

    Snow    Shadow   Cloud   Cirrus   Dilated cloud
    0/1     0/1      0/1     0/1      0/1

Here:

- `0` means the condition is **not present**
- `1` means the condition **is present**

For example:

- bit `3` represents **cloud**
- bit `4` represents **cloud shadow**
- bit `5` represents **snow**

### What does `1 << 3` mean?

Start with the binary number:

    00001

The operator `<<` means **shift to the left**.

So:

    1        → 00001
    1 << 1   → 00010
    1 << 2   → 00100
    1 << 3   → 01000

Therefore:

`1 << 3`

creates a number where **only bit 3 is switched on**.

Since bit 3 represents cloud, we have created a small mask that allows us to check only the **cloud switch**.

### What does `bitwiseAnd()` do?

Now consider:

`qa.bitwiseAnd(1 << 3)`

This asks:

> Is bit 3 switched on in this pixel's `QA_PIXEL` value?

For example, suppose a pixel has:

    QA_PIXEL:    01000
    Cloud mask:  01000
                 -----
    Result:      01000

The result is not zero, so **bit 3 is switched on**.

Therefore, this pixel is flagged as cloudy.

Now suppose another pixel has:

    QA_PIXEL:    00000
    Cloud mask:  01000
                 -----
    Result:      00000

The result is zero.

Therefore, **bit 3 is not switched on**, so the pixel is not flagged as cloudy.

### Why do we use `.eq(0)`?

Our objective is to keep pixels that are **not cloudy**.

Therefore:

`qa.bitwiseAnd(1 << 3).eq(0)`

can be read as:

> Check the cloud bit and keep the pixel if the result is zero.

In simple terms:

    Cloud bit = 0  → keep the pixel
    Cloud bit = 1  → remove the pixel

So the complete expression:

`qa.bitwiseAnd(1 << 3).eq(0)`

simply means:

> **Keep pixels that are not flagged as cloud.**

The same logic is used for the other quality flags.

For example:

`qa.bitwiseAnd(1 << 4).eq(0)`

means:

> **Keep pixels that are not flagged as cloud shadow.**

and:

`qa.bitwiseAnd(1 << 5).eq(0)`

means:

> **Keep pixels that are not flagged as snow.**

Therefore, when we write:

    clear = (
        qa.bitwiseAnd(1 << 1).eq(0)
        .And(qa.bitwiseAnd(1 << 2).eq(0))
        .And(qa.bitwiseAnd(1 << 3).eq(0))
        .And(qa.bitwiseAnd(1 << 4).eq(0))
        .And(qa.bitwiseAnd(1 << 5).eq(0))
    )

we are essentially saying:

> **Keep the pixel only if none of these unwanted quality flags are switched on.**

The important idea is:

**One `QA_PIXEL` value contains several yes/no quality switches. Bit operations allow us to inspect each switch separately.**

You do not need to memorise the bit numbers. Always check the dataset documentation to see what each bit represents.

In [ ]:
landsat_lst = landsat_hot_season.map(
    prepare_landsat_lst
)

lst = (
    landsat_lst
    .median()
    .clip(bengaluru)
)

lst

## 5. Map the median pre-monsoon LST

We now have one raster where each valid pixel represents the median surface temperature across the selected Landsat observations.


In [ ]:
lst_vis = {
    "min": 20,
    "max": 45,
    "palette": [
        "navy",
        "blue",
        "cyan",
        "yellow",
        "orange",
        "red",
        "darkred"
    ]
}

Map = geemap.Map()

Map.centerObject(bengaluru, 10)

Map.addLayer(
    lst,
    lst_vis,
    "Median LST: Mar–May 2025"
)

Map.addLayer(
    bengaluru,
    {"color": "white"},
    "Study area"
)

Map

## 6. Basic temperature statistics

We can calculate summary statistics across the study area using `reduceRegion()`.

Here we calculate:

- minimum;
- mean; and
- maximum surface temperature.


In [ ]:
lst_stats = lst.reduceRegion(
    reducer=(
        ee.Reducer.min()
        .combine(
            ee.Reducer.mean(),
            sharedInputs=True
        )
        .combine(
            ee.Reducer.max(),
            sharedInputs=True
        )
    ),
    geometry=bengaluru,
    scale=30,
    maxPixels=1e9
)

lst_stats.getInfo()

## 7. What should count as a 'high heat' area?

To identify areas experiencing high land-surface temperature, we need to choose a threshold.

For this exercise, we will use a simple absolute threshold:

> **High heat = LST greater than 40°C**

This means that every pixel with a land-surface temperature above `35°C` will be classified as a high-heat pixel.

An absolute threshold is easy to understand and interpret:

```text
LST ≤ 40°C  → not classified as high heat

LST > 40°C  → high heat

In [ ]:
# Define the high-heat threshold
heat_threshold = 40

# Identify pixels where LST is greater than 35°C
high_heat = (
    lst
    .gt(heat_threshold) ## greater than
    .selfMask()
    .rename("High_Heat")
)

In [ ]:

Map = geemap.Map()

Map.centerObject(
    bengaluru,
    10
)

# Add the complete LST surface
Map.addLayer(
    lst,
    lst_vis,
    "Land Surface Temperature"
)

# Add only pixels above 40°C
Map.addLayer(
    high_heat,
    {"palette": ["red"]},
    "LST > 40°C"
)

# Add Bengaluru boundary
Map.addLayer(
    bengaluru,
    {"color": "white"},
    "Bengaluru Boundary"
)

Map

## 8. Exposure requires population data

The heat raster tells us:

> **where the surface is hot.**

It does not tell us:

> **how many people live there.**

For that we add a population raster.

We use **WorldPop**, which estimates residential population in approximately 100 m grid cells.

For this teaching example, we use the **2020 population surface**.

### Important methodological note

Our heat analysis uses 2025 imagery, whereas this WorldPop product provides the population surface for 2020.

Therefore, the result should be described as an **illustrative exposure overlay**, not an exact 2025 population estimate.


In [ ]:
population = (
    ee.ImageCollection(
        "WorldPop/GP/100m/pop"
    )
    .filterBounds(bengaluru)
    .filterDate(
        "2020-01-01",
        "2021-01-01"
    )
    .mosaic()
    .select("population")
    .clip(bengaluru)
)

population

## Understanding `mosaic()`

`mosaic()` combines multiple images in an `ImageCollection` into **one single image**.

It is useful when different images cover different parts of the study area and we want one continuous raster.

### What happens when images overlap?

If several images contain a valid value for the **same pixel location**, `mosaic()` does **not** calculate an average.

Instead, Earth Engine gives priority to the images based on their order in the collection.

> The **last image** in the collection has the highest priority.

For example:

```text
Image 1 → 100
Image 2 → 120
Image 3 → 140

mosaic() → 140

In [ ]:
import geemap.colormaps as cm

population_vis = {
    "min": 0,
    "max": 100,
    "palette": cm.get_palette(
        "YlOrRd",
        n_class=6
    )
}

Map = geemap.Map()

Map.centerObject(
    bengaluru,
    10
)

Map.addLayer(
    population,
    population_vis,
    "WorldPop 2020"
)

Map.addLayer(
    high_heat,
    {"palette": ["cyan"]},
    "LST > 40°C"
)

Map.addLayer(
    bengaluru,
    {"color": "white"},
    "Bengaluru Boundary"
)

Map

## 9. Estimate population living in high-heat areas

First, calculate the total estimated population in the study area.

Then mask the population raster using the high-heat raster.

Only population cells overlapping the hottest 10% of the LST surface remain.


In [ ]:
# Calculate the total estimated population in the Bengaluru study area
total_population = ee.Number(
    population.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=bengaluru,
        scale=100,
        maxPixels=1e9
    ).get("population")
)

# Keep population values only where LST is greater than 40°C
population_high_heat = (
    population
    .updateMask(high_heat)
)

# Calculate the estimated population living in areas above 40°C
exposed_population = ee.Number(
    population_high_heat.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=bengaluru,
        scale=100,
        maxPixels=1e9
    ).get("population")
)

# Calculate the percentage of the total population exposed to LST > 40°C
exposure_share = (
    exposed_population
    .divide(total_population)
    .multiply(100)
)

# Display the results
print(
    "Estimated total population:",
    round(total_population.getInfo())
)

print(
    "Estimated population in areas with LST > 40°C:",
    round(exposed_population.getInfo())
)

print(
    "Share of population in areas with LST > 40°C:",
    round(exposure_share.getInfo(), 2),
    "%"
)

## Estimating Population Exposure to High Land-Surface Temperature

We now combine two raster datasets:

- the **WorldPop population raster**; and
- the **high-heat raster**, where pixels with `LST > 40°C` have been identified.

The objective is to estimate:

1. the total population within the Bengaluru study area;
2. the population located in areas where LST exceeds 40°C; and
3. the percentage of the total population located in these high-heat areas.

### 1. Calculate the total population

```python
total_population = ee.Number(
    population.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=bengaluru,
        scale=100,
        maxPixels=1e9
    ).get("population")
)
```

`population.reduceRegion()` summarizes the population raster over a specified geographic area.

- `reducer=ee.Reducer.sum()` adds the population values of all raster cells.
- `geometry=bengaluru` restricts the calculation to the Bengaluru boundary.
- `scale=100` specifies an analysis scale of approximately 100 m, corresponding to the approximate spatial resolution of the WorldPop raster.
- `maxPixels=1e9` allows Earth Engine to process a large number of pixels if required.
- `.get("population")` extracts the result for the `population` band.
- `ee.Number()` stores the result as an Earth Engine numerical object.

The result is the estimated **total population within Bengaluru**.

---

### 2. Keep population only in high-heat areas

```python
population_high_heat = (
    population
    .updateMask(high_heat)
)
```

The `high_heat` raster was created earlier by identifying pixels where:

```text
LST > 40°C → high heat

LST ≤ 40°C → not high heat
```

`updateMask(high_heat)` applies this high-heat mask to the population raster.

Conceptually:

```text
Population raster
        +
High-heat raster
        ↓
Population only where LST > 40°C
```

Population cells outside the high-heat areas are masked and are therefore excluded from the next calculation.

---

### 3. Calculate the population in high-heat areas

```python
exposed_population = ee.Number(
    population_high_heat.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=bengaluru,
        scale=100,
        maxPixels=1e9
    ).get("population")
)
```

We again use `reduceRegion()` with `ee.Reducer.sum()`.

However, this time the input is `population_high_heat`, which contains population values only where LST exceeds 40°C.

The result therefore represents the estimated population located in **high-heat areas**.

---

### 4. Calculate the share of population in high-heat areas

```python
exposure_share = (
    exposed_population
    .divide(total_population)
    .multiply(100)
)
```

First:

```python
exposed_population.divide(total_population)
```

calculates the proportion of the total population located in high-heat areas.

The calculation is:

$$
\text{Exposure Share}
=
\frac{\text{Population in areas with LST > 40°C}}
{\text{Total population}}
\times 100
$$

`.multiply(100)` converts the proportion into a percentage.

---

### 5. Bring the results back to Python

```python
print(
    "Estimated total population:",
    round(total_population.getInfo())
)

print(
    "Estimated population in areas with LST > 40°C:",
    round(exposed_population.getInfo())
)

print(
    "Share of population in areas with LST > 40°C:",
    round(exposure_share.getInfo(), 2),
    "%"
)
```

`.getInfo()` brings the small numerical result from the Earth Engine server into the local Python environment.

`round()` is used only to make the displayed results easier to read.

The final outputs therefore report:

- estimated total population;
- estimated population located in areas where `LST > 40°C`; and
- the percentage of the population located in these high-heat areas.

### Interpretation

This provides an estimate of **population exposure to high land-surface temperature**.

It does not by itself measure vulnerability or an individual's actual heat exposure. Heat risk can also depend on factors such as housing conditions, age, income, occupation, access to cooling, vegetation, and the amount of time people spend at different locations.

## Combining Raster Datasets with Different Spatial Resolutions

In this analysis, the two rasters have different pixel sizes:

- **Landsat high-heat raster** → approximately 30 m pixels
- **WorldPop population raster** → approximately 100 m pixels

So one WorldPop pixel covers the area of several smaller Landsat pixels.

Conceptually:

30 m heat pixels

Hot   Hot   Not hot  
Hot   Not   Not hot  
Hot   Hot   Not hot  

↓  

one larger 100 m population pixel

Population = 80 people

### What happens when we use `updateMask()`?

When we write:

`population_high_heat = population.updateMask(high_heat)`

Earth Engine has to place the 30 m heat mask onto the grid being used for the population calculation.

Earth Engine therefore **aligns/resamples the heat mask during the computation**.

By default, ordinary reprojection in Earth Engine uses **nearest-neighbour resampling** unless another method is specified.

So the simplified logic is:

30 m high-heat raster  
↓  
sampled onto the population grid  
↓  
100 m population cells  
↓  
Hot → keep the population value  
Not hot → mask the population value

For example, if a 100 m WorldPop pixel contains:

`Population = 80`

and the sampled heat-mask value is:

`1 = Hot`

then the population value of `80` is retained.

If the sampled heat-mask value is:

`0 = Not hot`

then that population cell is excluded from the exposed-population calculation.

### What does `scale=100` do later?

When we calculate:

`reduceRegion(reducer=ee.Reducer.sum(), geometry=bengaluru, scale=100)`

we are asking Earth Engine to perform the final population summation at an analysis scale of approximately **100 m**.

This is appropriate because the WorldPop raster itself is approximately 100 m resolution.

### Important limitation

This is a **simplified exposure estimate**.

Suppose one 100 m population cell contains 80 people, but the smaller 30 m heat pixels inside it look like this:

Hot   Hot   Not hot  
Hot   Not   Not hot  
Hot   Hot   Not hot  

Five out of nine smaller pixels are hot.

Our simple `updateMask()` approach does **not** explicitly calculate:

`5 / 9 = 56% of the population cell is hot`

Instead, the heat raster is sampled onto the population grid and the population cell is effectively treated as either **kept or masked**.

### A more refined approach

For a more detailed analysis, we could first aggregate the smaller 30 m heat pixels to the 100 m population grid.

Because the heat mask contains:

`1 = hot`  
`0 = not hot`

taking the **mean** of the smaller pixels gives the proportion of the larger cell that is hot.

For example:

`5 hot pixels out of 9`

`Hot fraction = 5 / 9 = 0.56`

If the 100 m population cell contains 80 people:

`Estimated exposed population = 80 × 0.56 ≈ 45 people`

Therefore:

Simple approach:  
`resample the heat mask → keep or mask population cells`

More detailed approach:  
`aggregate 30 m heat pixels → calculate hot fraction → multiply by population`

For the present exercise, the simpler method is useful for demonstrating the main idea:

**Heat hazard + population distribution → estimate of population exposure**

The key lesson is that **spatial resolution matters whenever raster datasets with different pixel sizes are combined**.

## 10. Hazard, exposure, and vulnerability are different

This is an important urban-analysis distinction.

### Hazard

Where is the physical environmental condition high?

Here: high land-surface temperature.

### Exposure

Who or what is located in the hazardous area?

Here: population residing in high-LST cells.

### Vulnerability

Who is more likely to suffer adverse consequences?

This may depend on:

- age;
- income;
- housing quality;
- access to cooling;
- health;
- occupation;
- green-space access; and
- other social and infrastructural conditions.

Satellite data alone generally cannot tell us all of these things.

This is why environmental-risk analysis often combines remote sensing with socioeconomic and administrative data.


# Class Exercise 20 — Urban Thermal Stress and Population Exposure Using UTCI

Choose **any city other than Bengaluru**. You may select your hometown or another city of interest.

The objective is to examine urban thermal stress during one warm season using the Universal Thermal Climate Index (UTCI), and then estimate the population exposed to high thermal stress.

## Data Source 1 — UTCI

Use the **GloUTCI-M** dataset available in Google Earth Engine:

`projects/sat-io/open-datasets/gloutci-m`

The dataset provides:

- monthly UTCI values;
- approximately **1 km spatial resolution**;
- global coverage;
- data from **2000 to 2022**.

The stored values must be **divided by 100** to obtain UTCI in °C.

## Data Source 2 — Population

Use the WorldPop population dataset:

`WorldPop/GP/100m/pop`

The population raster has a spatial resolution of approximately **100 m**.

Use population data from the same or closest available year to your UTCI analysis.

## Tasks

1. Select a city **other than Bengaluru** and obtain its boundary.

2. Select one year of dat.

3. Select one warm season appropriate for your city.

   For an Indian city, you may use:

   **March–May**

4. Load the GloUTCI-M dataset and filter it for:

   - your city;
   - your selected year; and
   - your selected season.

5. Convert the UTCI values to °C by dividing them by `100`.

6. Calculate the **mean UTCI** for the selected season.

7. Create a map showing the seasonal mean UTCI across the city.

8. Identify areas experiencing high thermal stress.

   For this exercise, use:

   `UTCI ≥ 32°C`

   This represents **strong heat stress or higher**.

9. Load the WorldPop population raster.

10. Estimate:

   - the total population of the city;
   - the population located in areas where `UTCI ≥ 32°C`; and
   - the percentage of the total population exposed to strong heat stress or higher.

Calculate:

$$
\text{Population Exposure (\%)} =
\frac{\text{Population in areas with UTCI} \geq 32^\circ C}
{\text{Total Population}}
\times 100
$$

11. Produce a map showing the areas with high UTCI together with the population distribution.

12. Write a short interpretation of your results.

Discuss:

- the spatial pattern of UTCI across the city;
- which areas experience relatively higher thermal stress;
- the estimated population exposed to `UTCI ≥ 32°C`; and
- any limitations of the analysis.

## Important Note on Spatial Resolution

The two datasets have different spatial resolutions:

- **UTCI:** approximately 1 km
- **WorldPop:** approximately 100 m

Therefore, several smaller population pixels fall within one larger UTCI pixel.

This means that people living within the same 1 km UTCI pixel are assigned approximately the same thermal-stress value.

The results should therefore be interpreted as a **city-scale estimate of population exposure**, rather than thermal stress at the street or individual level.

## UTCI and LST Are Different

**Land Surface Temperature (LST)** tells us how hot the Earth's surface is.

**UTCI** represents the thermal stress experienced by people by considering atmospheric conditions such as temperature, humidity, wind, and radiation.

Therefore:

**LST → How hot is the surface?**

**UTCI → How stressful are the outdoor thermal conditions for people?**

## A reusable Earth Engine thinking framework

Before starting any Earth Engine analysis, ask:

1. **What is my research question?**
2. **What physical or social variable do I need?**
3. **Which dataset measures or estimates that variable?**
4. **What is its spatial resolution?**
5. **What is its temporal resolution?**
6. **What quality masking or preprocessing is required?**
7. **What does the variable actually mean?**
8. **What claims can I reasonably make from it?**

This is more important than memorising Earth Engine commands.


## Dataset notes

The notebook uses the following Earth Engine collections:

- `COPERNICUS/S2_SR_HARMONIZED` — Sentinel-2 Level-2A Surface Reflectance
- `LANDSAT/LC08/C02/T1_L2` — Landsat 8 Collection 2 Level-2
- `LANDSAT/LC09/C02/T1_L2` — Landsat 9 Collection 2 Level-2
- `WorldPop/GP/100m/pop` — WorldPop estimated residential population

Dataset names, band names, scale factors, quality flags, and units should always be checked in the **Google Earth Engine Data Catalog** before using a dataset in a new analysis.
